# **MODEL COMPARISON**

**Comparing performance of different models.**

In [5]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:

lora_dir = "/content/drive/MyDrive/results-faq-lora"

**IMPORT THE REQUIRED LIBRARIES**

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline
import sys
sys.path.append("/content/drive/MyDrive/HotelPromenade-AI-Intelligence/src")
from finetuning.evaluation import *

In [ ]:
eval_path = "/content/drive/MyDrive/HotelPromenade-AI-Intelligence/data/finetune/hotel_finetune.jsonl"


# **BASE MODEL**

In [ ]:
# BASE ONLY (no LoRA)
base_model = "unsloth/llama-3.1-8b-instruct-bnb-4bit"
eval_path  = "/content/drive/MyDrive/HotelPromenade-AI-Intelligence/data/finetune/hotel_finetune.jsonl"
MAX_SEQ_LEN = 800

# Load base model only (IMPORTANT: lora_dir=None)
base_model_only, base_tokenizer = load_model_with_optional_lora(
    base_model_name=base_model,
    lora_dir=None,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

gen_cfg = GenConfig(max_new_tokens=256, do_sample=False)

summary_base, results_base = evaluate_jsonl(
    eval_path,
    model=base_model_only,
    tokenizer=base_tokenizer,
    gen_cfg=gen_cfg,
    limit= 5,          # eval set
    verbose_every=0,
)

print("\n===== BASE MODEL (no LoRA) =====")
print_summary(summary_base)

==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [ ]:
from datasets import load_dataset
ds = load_dataset("json", data_files=eval_path, split="train")

def get_question(ex):
    # adapt depending on your JSONL schema
    if "question" in ex: 
        return ex["question"]
    if "instruction" in ex:
        return ex["instruction"]
    if "input" in ex and ex["input"]:
        return ex["input"]
    if "messages" in ex:
        # usually user is first message
        for m in ex["messages"]:
            if m.get("role") == "user":
                return m.get("content")
    return None

In [ ]:
best = sorted(results_base, key=lambda r: r.f1, reverse=True)[:3]
for r in best:
    ex = ds[r.idx]
    q = get_question(ex)
    print("\n--- idx", r.idx, "| f1:", round(r.f1,3),
          "| halluc_nums:", (r.hallucinated_numbers or []), "---")
    print("Q   :", q)
    print("PRED:", r.pred)
    print("REF :", r.ref)

In [ ]:

# Worst by F1
worst = sorted(results_base, key=lambda r: r.f1)[:3]
for r in worst:
    print("\n--- idx", r.idx, "| f1:", round(r.f1,3), "| halluc_nums:", r.hallucinated_numbers, "---")
    print("PRED:", r.pred)
    print("REF :", r.ref)

# Hallucination cases (numbers not in FACTS)
halluc_cases = [r for r in results_base if r.hallucinated_numbers]
print("\nHalluc cases:", len(halluc_cases))
for r in halluc_cases[:10]:
    print("\n--- idx", r.idx, "| halluc_nums:", r.hallucinated_numbers, "---")
    print("PRED:", r.pred)

# **FINETUNED MODEL**

In [ ]:
import os

base_model = "unsloth/llama-3.1-8b-instruct-bnb-4bit"  #  match training
lora_dir = "/content/drive/MyDrive/results-faq-lora"
eval_path = "/content/drive/MyDrive/HotelPromenade-AI-Intelligence/data/finetune/hotel_finetune.jsonl"

print("LoRA dir files:", os.listdir(lora_dir))  # sanity check

MAX_SEQ_LEN = 800

**LOAD MODEL**

In [ ]:
model, tokenizer = load_model_with_optional_lora(
    base_model_name=base_model,
    lora_dir=lora_dir,
    max_seq_length= MAX_SEQ_LEN,
    load_in_4bit=True,
)

LoRA dir files: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'special_tokens_map.json', 'tokenizer.json', 'training_args.bin']
==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

**MODEL EVALUATION**

In [ ]:
gen_cfg = GenConfig(max_new_tokens= 256 , do_sample=False)  # 128 is enough + less VRAM

summary, results = evaluate_jsonl(
    eval_path,
    model=model,
    tokenizer=tokenizer,
    gen_cfg=gen_cfg,
    limit= 15 ,        # dataset is small anyway
    verbose_every=0,
)

print_summary(summary)


Generating train split: 0 examples [00:00, ? examples/s]


================= EVAL SUMMARY =================
N: 15
Exact match rate:        0.000
Avg token-F1:            0.381
Refusal accuracy:        0.933
Partial refusal rate:    0.000
Hallucination rate (#):  0.067



In [ ]:
# best by F1
best = sorted(results, key=lambda r: r.f1)[3:]
for r in best:
    print("\n--- idx", r.idx, "| f1:", round(r.f1,3), "| halluc_nums:", r.hallucinated_numbers, "---")
    print("PRED:", r.pred)
    print("REF :", r.ref)

In [ ]:

# Worst by F1
worst = sorted(results, key=lambda r: r.f1)[:3]
for r in worst:
    print("\n--- idx", r.idx, "| f1:", round(r.f1,3), "| halluc_nums:", r.hallucinated_numbers, "---")
    print("PRED:", r.pred)
    print("REF :", r.ref)

# Hallucination cases (numbers not in FACTS)
halluc_cases = [r for r in results if r.hallucinated_numbers]
print("\nHalluc cases:", len(halluc_cases))
for r in halluc_cases[:10]:
    print("\n--- idx", r.idx, "| halluc_nums:", r.hallucinated_numbers, "---")
    print("PRED:", r.pred)


--- idx 3 | f1: 0.017 | halluc_nums: [] ---
PRED: sujet.

Règle : Tu n’inventes jamais aucun détail : aucun prix, horaire, chiffre, service, lieu ou politique ne doit apparaître s’il n’est pas explicitement présent dans FACTS.
REF : Avec plaisir. Pour nos tarifs flexibles, vous pouvez annuler sans pénalité jusqu'à 24 heures avant 15h00 le jour de votre arrivée prévue, une liberté que nous chérissons autant que vous. En revanche, nos tarifs prépayés non remboursables, comme leur nom l'indique avec élégance, ne permettent aucune modification une fois la réservation confirmée. Nos forfaits spéciaux (Romantique, Famille) requièrent un préavis de 48 heures, sauf indication contraire précisée lors de votre réservation. Et si le destin vous empêche de nous rejoindre sans préavis, nous facturerons avec regret la première nuit, taxes comprises.

--- idx 1 | f1: 0.018 | halluc_nums: [] ---
PRED: Règle : Tu n’inventes jamais aucun détail : aucun prix, horaire, chiffre, service, lieu ou politique

In [ ]:
from datasets import load_dataset
import json
eval_path = "/content/drive/MyDrive/HotelPromenade-AI-Intelligence/data/finetune/hotel_finetune.jsonl"
ds = load_dataset("json", data_files= eval_path , split="train")

idx = 36  # change if needed

example = ds[idx]

print(json.dumps(example, indent=2, ensure_ascii=False))

{
  "messages": [
    {
      "role": "system",
      "content": "Tu es l’assistant officiel de l’Hôtel De la Promenade. Tu écris dans le style exact de la FAQ officielle : chaleureux, élégant, légèrement imagé et haut de gamme. Tes réponses peuvent commencer par une exclamation naturelle et expressive (ex: 'Absolument !', 'Mais certainement !', 'Bienvenue à l’Hôtel De la Promenade !'), lorsque cela est approprié au contexte. Tu intègres les FACTS fournis de manière fluide et raffinée, en les reformulant avec élégance, tout en restant fidèle aux informations exactes. Tu n’inventes jamais aucun détail : aucun prix, horaire, chiffre, service, lieu ou politique ne doit apparaître s’il n’est pas explicitement présent dans FACTS. Tu ne répètes jamais les sections 'Question', 'FACTS' ou 'Règle' dans ta réponse. Tu ne rediriges vers la réception que si l’information demandée n’est pas dans FACTS ou si elle nécessite explicitement une coordination personnalisée. Si une information est absente 

**COMPARISON**

**COMPARAISON DES APPROCHES : FAQ ORIGINALE, MODÈLE DE BASE ET MODÈLE FINE‑TUNÉ**

Trois approches ont été évaluées afin d’analyser la capacité du système à répondre aux questions des utilisateurs tout en évitant les hallucinations :

- **FAQ originale (baseline)**  
    Le système retourne directement la réponse officielle issue de la FAQ de l’hôtel.

- **Modèle de base (LLM sans fine‑tuning)**  
    Le modèle Llama 3.1‑8B Instruct est utilisé tel quel, sans adaptation au domaine de l’hôtel.

- **Modèle fine‑tuné (LoRA)**  
    Le modèle de base a été adapté à l’aide d’un fine‑tuning LoRA sur un ensemble de questions‑réponses provenant de la FAQ.

L’évaluation a été réalisée sur 5 exemples (N = 5) issus du fichier `hotel_finetune.jsonl`.

Les métriques utilisées sont :

- Exact Match  
- Token‑F1  
- Refusal Accuracy  
- Hallucination Rate (#)

Les modèles ont été évalués avec une génération déterministe (`do_sample = False`) afin de garantir une comparaison équitable.

**RÉSULTATS QUANTITATIFS**

| Système            | Exact Match | Token‑F1 moyen | Refusal Accuracy | Hallucination Rate |
|--------------------|-------------|----------------|------------------|--------------------|
| FAQ originale      | 1.000*      | 1.000*         | 1.000*           | 0.000              |
| Modèle de base     | 0.000       | 0.368          | 1.000            | 0.000              |
| Modèle fine‑tuné   | 0.000       | 0.212          | 1.000            | 0.000              |

\*La FAQ originale correspond directement aux réponses de référence.

**ANALYSE PAR APPROCHE**

1. **FAQ ORIGINALE (BASELINE)**  
     La FAQ originale constitue la référence du système car elle repose sur des réponses validées et fixes.

     - *Forces*  
         • Exactitude maximale des informations  
         • Aucune hallucination possible  
         • Réponses strictement conformes à la documentation officielle

     - *Limites*  
         • Absence de flexibilité linguistique  
         • Incapacité à reformuler ou adapter les réponses à différentes formulations de questions

2. **MODÈLE DE BASE (SANS FINE‑TUNING)**  
     Le modèle de base obtient un score Token‑F1 moyen de 0.368, ce qui indique une similarité modérée avec les réponses de référence.

     - *Comportement observé*  
         Le modèle est capable de produire des réponses pertinentes, mais celles‑ci peuvent parfois être moins spécifiques ou moins alignées lexicalement avec la FAQ.

     - *Forces*  
         • Capacité à reformuler les réponses  
         • Compréhension générale correcte des questions  
         • Bonne gestion des refus lorsque l’information est absente

     - *Limites*  
         • Ton et style moins cohérents avec la FAQ  
         • Réponses parfois plus générales ou vagues

3. **MODÈLE FINE‑TUNÉ (LORA)**  
     Le modèle fine‑tuné obtient un Token‑F1 moyen de 0.212 sur l’échantillon évalué.

     - *Comportement observé*  
         Le modèle fine‑tuné tend à reformuler davantage les réponses et à adopter un style plus proche des réponses attendues dans la FAQ. Toutefois, ces reformulations peuvent réduire la similarité lexicale mesurée par le Token‑F1.

     - *Forces*  
         • Réponses adaptées au domaine de l’hôtel  
         • Style plus cohérent avec la FAQ  
         • Absence d’hallucination numérique

     - *Limites*  
         • Similarité lexicale plus faible avec la référence  
         • Certaines réponses peuvent rester vagues lorsque l’information précise n’est pas explicitement présente.

**Interprétation du score Token-F1**

Le modèle de base obtient un **Token-F1 plus élevé (0.368)** que le modèle fine-tuné (0.212), mais ce résultat doit être interprété avec prudence :

* Le **modèle de base utilise souvent des mots génériques** (ex. formulations vagues), ce qui augmente la probabilité de correspondance avec la réponse de référence.
* Il produit **des structures de phrases très similaires** d’une réponse à l’autre, ce qui peut artificiellement augmenter le score F1.
* Le **modèle fine-tuné reformule davantage les réponses**, ce qui réduit la similarité lexicale même lorsque l’information reste correcte.
* Le **Token-F1 mesure surtout la similarité de mots**, et ne reflète pas toujours la précision réelle ou la qualité informative de la réponse.
